# Transform annotations to a pandas dataframe.

Use this notebook to transform annotations in gpkg files to a pandas dataframe in a .parquet file.

In [2]:
import glob
import pandas as pd
import geopandas as gpd
import rasterio
from satellite_images_nso_datascience.nso_ds_classes.nso_ds_normalize_scaler import scaler_class_all
from satellite_images_nso_datascience.annotations.data_preparation import extract_dataframe_pixels_values_from_tif_and_polygons
from satellite_images_nso_datascience.annotations.utils import get_scaler_filepath
from satellite_images_nso_datascience.annotations.data_loader import load_annotations_polygons, load_annotations_polygons_gpkg
import os
from satellite_images_nso_datascience.other import functions
import settings_blob
from rasterio.mask import mask

import rasterio
from rasterio.crs import CRS

In [19]:
#annotation_folder_path = "C:/repos/satellite-images-nso-datascience/data/annotations/Lansingerland/Annotations_begroeiing_Lansingerland_sat_images.gpkg"
annotation_folder_path = "C:/repos/satellite-images-nso-datascience/data/annotations/Lansingerland/Annotations_begroeiing_Lansingerland_sat_images_shadow_classes.gpkg"
tif_files_path = "E:/data/langslingerland/TIF/" 

name_table = annotation_folder_path.split("/")[-1].split(".")[0]

In [23]:

tif_file = "E:/data/langslingerland/sat_images/Multiband/37FN1_H8.tiff"

annotations_polygons_gdf = load_annotations_polygons_gpkg(annotation_folder_path)

annotations_polygons_gdf.columns = ["Label", "geometry", "name"]
annotations_polygons_gdf = annotations_polygons_gdf.drop(["name"],axis=1)

with rasterio.open(tif_file, "r+") as dataset:
    # Define the correct CRS (example: EPSG:4326)
    new_crs = CRS.from_epsg(28992)
    dataset.crs = new_crs

    

    df = extract_dataframe_pixels_values_from_tif_and_polygons(
        dataset,
        annotations_polygons_gdf
    )

    if len(df) ==0:
        print(" Data frame is empty!")


Found some empty pixels!


In [24]:
df['label'].value_counts()

label
anders             618494
gras               180492
Anders              40752
Schaduw             15960
boom                 5730
grass                5040
Gras                 4288
Schaduw anders       3761
Schaduw gras         1304
Boom                 1128
lage begroeiing       898
Name: count, dtype: int64

In [35]:
df['label'] = df['label'].replace("Amders", "Anders") 
df['label'] = df['label'].replace("grass", "Gras") 
df['label'] = df['label'].replace("gras", "Gras") 
df['label'] = df['label'].replace("anders", "Anders") 
df['label'] = df['label'].replace("boom", "Boom") 
df['label'] = df['label'].replace("lage begroeiing", "Lage begroeiing") 

In [36]:
df = df[df['R'] != -9999.0]
df = df[df['label'] != "Schaduw"]

In [37]:
df['label'].value_counts()

label
Anders             318303
Gras                92020
Boom                 4009
Schaduw anders       1365
Schaduw gras          660
Lage begroeiing       212
Name: count, dtype: int64

In [39]:
df['label'].unique()

array(['Gras', 'Anders', 'Boom', 'Lage begroeiing', 'Schaduw gras',
       'Schaduw anders'], dtype=object)

In [38]:
df.to_parquet("Lansingerland/begroeiing_annotations_pixels_sat_images.parquet")